# Floci masterclass — live walkthrough

This notebook drives the two demos **one AWS call at a time**, so every
mechanism stays visible: no `deploy.py`/`run_demo.py` black boxes here,
just the raw `boto3` calls in the order Floci actually executes them.

**Before you start:** run `make up` from a terminal (once per session).
It starts Floci and this notebook's kernel container.

In [ ]:
import os
from pathlib import Path

# Euporie and `jupyter execute` may start the kernel's cwd at either the
# repo root or this notebook's own directory, depending on how they're
# launched. Normalize to the repo root so every relative path below
# (demos/...) resolves the same way either way.
if not Path("demos").exists():
    os.chdir(Path.cwd().parent)

Path.cwd()

## 0. Connect to Floci

In [ ]:
import json
import os
import time
import urllib.parse

import boto3
from botocore.exceptions import ClientError

ENDPOINT = os.environ.get("FLOCI_ENDPOINT", "http://floci:4566")
ENDPOINT

In [ ]:
import urllib.request

health = urllib.request.urlopen(f"{ENDPOINT}/_floci/health")
json.loads(health.read())["version"]

---
## Demo 1 — S3 upload triggers a Lambda

**Goal:** dropping a CSV under `in/` in bucket `demo` should automatically
run a Lambda that adds a `prediction` column and writes the result under
`out/`. Nothing here is invoked manually — the trigger is the S3 event.

### 1.1 — One boto3 client per service

In [ ]:
iam = boto3.client("iam", endpoint_url=ENDPOINT)
lam = boto3.client("lambda", endpoint_url=ENDPOINT)
s3 = boto3.client("s3", endpoint_url=ENDPOINT)

ROLE_NAME = "lambda-exec-role"
FUNC_NAME = "predict"
BUCKET = "demo"

### 1.2 — IAM role the Lambda will run as

Trust policy: only the Lambda service may assume this role.

In [ ]:
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

try:
    role = iam.create_role(
        RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy)
    )
    role_arn = role["Role"]["Arn"]
except iam.exceptions.EntityAlreadyExistsException:
    role_arn = iam.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]

role_arn

### 1.3 — Least-privilege execution policy

Read from `demo/in/*`, write to `demo/out/*`, log to CloudWatch. That's
the whole footprint the handler needs — no `AdministratorAccess`.

In [ ]:
execution_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject"],
            "Resource": [f"arn:aws:s3:::{BUCKET}/in/*"],
        },
        {
            "Effect": "Allow",
            "Action": ["s3:PutObject"],
            "Resource": [f"arn:aws:s3:::{BUCKET}/out/*"],
        },
        {
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": "arn:aws:logs:*:*:*",
        },
    ],
}

iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName="predict-execution-policy",
    PolicyDocument=json.dumps(execution_policy),
)
print("Policy attached.")

### 1.4 — Package the handler

The handler lives at
`../demos/01-s3-lambda-notification/lambda_predict/handler.py`. Zip it up
exactly as Lambda expects (module at the zip root).

In [ ]:
import zipfile
from pathlib import Path

handler_path = Path("demos/01-s3-lambda-notification/lambda_predict/handler.py")
zip_path = Path("demos/01-s3-lambda-notification/predict.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(handler_path, "handler.py")

zip_bytes = zip_path.read_bytes()
f"{len(zip_bytes)} bytes"

### 1.5 — Create the Lambda function

In [ ]:
try:
    fn = lam.create_function(
        FunctionName=FUNC_NAME,
        Runtime="python3.12",
        Role=role_arn,
        Handler="handler.handler",
        Code={"ZipFile": zip_bytes},
        Timeout=60,
        MemorySize=256,
    )
    func_arn = fn["FunctionArn"]
except lam.exceptions.ResourceConflictException:
    lam.update_function_code(FunctionName=FUNC_NAME, ZipFile=zip_bytes)
    func_arn = lam.get_function(FunctionName=FUNC_NAME)["Configuration"]["FunctionArn"]

func_arn

### 1.6 — Wait for it to become `Active`

Same lifecycle as real AWS Lambda: `Pending` while Floci provisions it,
then `Active`.

In [ ]:
for _ in range(30):
    conf = lam.get_function_configuration(FunctionName=FUNC_NAME)
    print(conf["State"])
    if conf["State"] == "Active":
        break
    time.sleep(2)

### 1.7 — Create the bucket the demo uploads into

In [ ]:
try:
    s3.create_bucket(Bucket=BUCKET)
except s3.exceptions.BucketAlreadyOwnedByYou:
    pass
print(f"Bucket '{BUCKET}' ready.")

### 1.8 — Let S3 invoke the Lambda

Without this resource-based permission, S3's own notification would be
silently rejected by Lambda — this is the piece people most often forget
on real AWS too.

In [ ]:
try:
    lam.add_permission(
        FunctionName=FUNC_NAME,
        StatementId="s3invoke",
        Action="lambda:InvokeFunction",
        Principal="s3.amazonaws.com",
        SourceArn=f"arn:aws:s3:::{BUCKET}",
    )
    print("Permission granted.")
except lam.exceptions.ResourceConflictException:
    print("Permission already present.")

### 1.9 — Wire the S3 event notification

In [ ]:
s3.put_bucket_notification_configuration(
    Bucket=BUCKET,
    NotificationConfiguration={
        "LambdaFunctionConfigurations": [
            {
                "LambdaFunctionArn": func_arn,
                "Events": ["s3:ObjectCreated:*"],
                "Filter": {"Key": {"FilterRules": [{"Name": "prefix", "Value": "in/"}]}},
            }
        ]
    },
)
print("Notification configured: in/* -> predict()")

---
### 1.10 — The actual trigger

Everything above was setup. **This cell is the only thing an end user
would ever do**: drop a file in the bucket.

In [ ]:
content = b"score\n0.9\n0.2\n0.7\n"

# Clear any leftover output from a previous run of this notebook so the
# timing below reflects this upload, not a stale file.
s3.delete_object(Bucket=BUCKET, Key="out/scores.csv")

upload_start = time.time()
s3.put_object(Bucket=BUCKET, Key="in/scores.csv", Body=content)
print("Uploaded in/scores.csv — watch for the Lambda to pick it up below.")

### 1.11 — Watch the output appear

On a cold Floci this can take ~30s (pulling the Lambda runtime image the
first time); it's fast afterwards. Re-run this cell if it times out.

In [ ]:
TIMEOUT_SECONDS = 180
deadline = upload_start + TIMEOUT_SECONDS
result = None

while time.time() < deadline:
    try:
        obj = s3.get_object(Bucket=BUCKET, Key="out/scores.csv")
        result = obj["Body"].read().decode("utf-8")
        break
    except s3.exceptions.NoSuchKey:
        time.sleep(2)

elapsed = time.time() - upload_start
print(f"out/scores.csv appeared after {elapsed:.2f}s\n")
print(result if result else "TIMEOUT — re-run this cell.")

**Takeaway:** the Lambda never runs unless something writes to
`in/*` in `demo` — no polling, no scheduler, purely event-driven, and the
role it runs as can only touch the two prefixes it needs.

---
## Demo 2 — IAM policy enforcement

**Goal:** show that Floci actually *enforces* IAM policies, not just
records them, when enforcement is turned on.

> **Before running this section:** in a separate terminal, run
> `make floci-iam-on`. It restarts Floci with
> `FLOCI_SERVICES_IAM_ENFORCEMENT_ENABLED=true` (a live cell above can't
> restart Floci itself — this notebook's container has no Docker access).
> Run `make floci-iam-off` afterwards if you want to go back to the
> permissive default for demo 1.

### 2.1 — Admin client (test/test bypasses enforcement)

In [ ]:
admin_iam = boto3.client(
    "iam", endpoint_url=ENDPOINT, aws_access_key_id="test", aws_secret_access_key="test"
)
USER = "junior"

### 2.2 — Reset `junior` to a clean slate

So this section behaves the same whether it's the first time you run it
today or the fifth: no leftover policy, no pile-up of old access keys.

In [ ]:
try:
    admin_iam.create_user(UserName=USER)
except admin_iam.exceptions.EntityAlreadyExistsException:
    pass

for policy_name in admin_iam.list_user_policies(UserName=USER)["PolicyNames"]:
    admin_iam.delete_user_policy(UserName=USER, PolicyName=policy_name)

for key in admin_iam.list_access_keys(UserName=USER)["AccessKeyMetadata"]:
    admin_iam.delete_access_key(UserName=USER, AccessKeyId=key["AccessKeyId"])

print(f"'{USER}' reset: no policies, no access keys.")

### 2.3 — Issue junior a fresh access key

In [ ]:
key = admin_iam.create_access_key(UserName=USER)
junior_access_key = key["AccessKey"]["AccessKeyId"]
junior_secret_key = key["AccessKey"]["SecretAccessKey"]

junior_access_key  # secret key stays in the variable, not printed on screen

In [ ]:
s3_as_junior = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    aws_access_key_id=junior_access_key,
    aws_secret_access_key=junior_secret_key,
)

### 2.4 — junior tries to list buckets (no policy yet)

**This is the slide cell.** Expect `403 AccessDenied`.

In [ ]:
try:
    s3_as_junior.list_buckets()
    print("UNEXPECTED SUCCESS — is FLOCI_SERVICES_IAM_ENFORCEMENT_ENABLED=true? "
          "Run `make floci-iam-on` in a terminal, then re-run this cell.")
except ClientError as e:
    status = e.response["ResponseMetadata"]["HTTPStatusCode"]
    code_ = e.response["Error"]["Code"]
    message = e.response["Error"]["Message"]
    print(f"{status} {code_}: {message}")

### 2.5 — Grant the missing permission

In [ ]:
allow_list_buckets = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": "s3:ListAllMyBuckets", "Resource": "*"}
    ],
}

attach_start = time.time()
admin_iam.put_user_policy(
    UserName=USER, PolicyName="AllowListBuckets", PolicyDocument=json.dumps(allow_list_buckets)
)
print("Inline policy attached.")

### 2.6 — junior tries again

Expect success this time — and near-instant, unlike real AWS where policy
propagation can take a few seconds.

In [ ]:
resp = s3_as_junior.list_buckets()
elapsed = time.time() - attach_start
print(f"Success {elapsed:.2f}s after attaching the policy.")
[b["Name"] for b in resp["Buckets"]]

**Takeaway:** enforcement is **off by default** in Floci — without
`FLOCI_SERVICES_IAM_ENFORCEMENT_ENABLED=true`, every call above would
have succeeded even for step 2.4. Policy propagation is instant here,
which is worth calling out as a difference from real AWS.

---
## Cleanup

Not run automatically from here — tear down from a terminal when you're
done presenting:

```bash
make clean
```